In [14]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

import os
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

In [15]:
X_train = pd.read_csv('../data/X_train_data.csv')
y_train = pd.read_csv('../data/Y_train_data.csv')
X_test = pd.read_csv('../data/X_test_data.csv')
y_test = pd.read_csv('../data/Y_test_data.csv')

X_train = X_train.values
y_train = y_train.values
X_test = X_test.values
y_test = y_test.values

X_train = torch.from_numpy(X_train).float()
y_train = torch.from_numpy(y_train).float()
X_test = torch.from_numpy(X_test).float()
y_test = torch.from_numpy(y_test).float()


In [16]:
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

In [17]:
print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)
print(X_val.shape)
print(y_val.shape)


torch.Size([7864, 84])
torch.Size([7864, 1])
torch.Size([2458, 84])
torch.Size([2458, 1])
torch.Size([1967, 84])
torch.Size([1967, 1])


In [30]:
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val, dtype=torch.float32)

/var/folders/hj/pwp8mnp90wb03qbkc2c60_500000gn/T/ipykernel_12335/2434944468.py:1: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
/var/folders/hj/pwp8mnp90wb03qbkc2c60_500000gn/T/ipykernel_12335/2434944468.py:2: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
/var/folders/hj/pwp8mnp90wb03qbkc2c60_500000gn/T/ipykernel_12335/2434944468.py:3: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X_val_tensor = torch.tensor(X_val, dtype

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import pandas as pd
import os
import numpy as np
from sklearn.model_selection import train_test_split

class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        # Input layer: 84 features
        self.fc1 = nn.Linear(84, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 64)
        # Output layer: 1 unit for regression
        self.fc4 = nn.Linear(64, 1)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))
        x = self.fc4(x)
        return x

def train_model():

    net = Net()
    criterion = nn.MSELoss()
    optimizer = optim.Adam(net.parameters(), lr=0.001)

    # 5. Training Loop
    epochs = 100
    batch_size = 64
    num_samples = X_train_tensor.shape[0]
    
    print("Starting training...")
    for epoch in range(epochs):
        net.train()
        permutation = torch.randperm(num_samples)
        
        train_loss = 0.0
        for i in range(0, num_samples, batch_size):
            indices = permutation[i:i+batch_size]
            batch_x, batch_y = X_train_tensor[indices], y_train_tensor[indices]
            
            optimizer.zero_grad()
            outputs = net(batch_x)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item() * batch_x.size(0)
            
        train_loss /= num_samples
        
        # Validation step
        net.eval()
        with torch.no_grad():
            val_outputs = net(X_val_tensor)
            val_loss = criterion(val_outputs, y_val_tensor).item()
            val_mae = torch.mean(torch.abs(val_outputs - y_val_tensor)).item()
            val_acc = (torch.abs(val_outputs - y_val_tensor) <= 1.0).float().mean().item() * 100

        if (epoch + 1) % 10 == 0:
            print(f"Epoch [{epoch+1}/{epochs}] | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val MAE: {val_mae:.4f} | Val Acc: {val_acc:.2f}%")

    return net


In [38]:
train_model()

Starting training...
Epoch [10/100] | Train Loss: 9.5988 | Val Loss: 8.5801 | Val MAE: 1.0664 | Val Acc: 72.70%
Epoch [20/100] | Train Loss: 61.4420 | Val Loss: 219.6347 | Val MAE: 3.6153 | Val Acc: 35.59%
Epoch [30/100] | Train Loss: 6.4486 | Val Loss: 3.5275 | Val MAE: 0.8818 | Val Acc: 71.63%
Epoch [40/100] | Train Loss: 2.0873 | Val Loss: 2.7605 | Val MAE: 0.8302 | Val Acc: 78.95%
Epoch [50/100] | Train Loss: 0.7893 | Val Loss: 0.7594 | Val MAE: 0.5856 | Val Acc: 84.70%
Epoch [60/100] | Train Loss: 0.5894 | Val Loss: 0.6494 | Val MAE: 0.5012 | Val Acc: 90.19%
Epoch [70/100] | Train Loss: 0.5179 | Val Loss: 0.6801 | Val MAE: 0.5702 | Val Acc: 85.36%
Epoch [80/100] | Train Loss: 4.7135 | Val Loss: 3.7815 | Val MAE: 1.1475 | Val Acc: 68.23%
Epoch [90/100] | Train Loss: 0.5041 | Val Loss: 0.5343 | Val MAE: 0.4977 | Val Acc: 89.48%
Epoch [100/100] | Train Loss: 0.5442 | Val Loss: 0.4236 | Val MAE: 0.4939 | Val Acc: 90.34%


Net(
  (fc1): Linear(in_features=84, out_features=256, bias=True)
  (fc2): Linear(in_features=256, out_features=128, bias=True)
  (fc3): Linear(in_features=128, out_features=64, bias=True)
  (fc4): Linear(in_features=64, out_features=1, bias=True)
)